In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_models, ModelTypes, model_names #, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask
from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
import dinosaw.utils as utils
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import TypeAlias


SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

flash attention installed


In [2]:
selected_models: tuple[ModelTypes, ...] = ('dv_b', 'dv2_b', 'dv3_b', 'vit_b', 'vit_b_in', 'clip_b' )

models = get_models(selected_models, '../../trained_models', device=DEVICE,  conf_path='../../dinov3')
S = models[selected_models[0]].stride

n_layers = 12
n_dims = 768

dinov3_vitb16 dinov3_vitb_patch16_reg4.pth ../../dinov3


In [3]:
def get_feature_list(
    model: PretrainedViTWrapper,
    pil_img: Image.Image,
    channel_last: bool = False,
    to_half: bool = False,
    device: str = "cuda:0",
    N: int = 11
) -> list[np.ndarray]:
    tr = utils.closest_resize(pil_img.height, pil_img.width, model.stride)
    img_tensor = utils.convert_image(pil_img, tr, device_str=device, to_half=to_half)

    is_dv3 = 'dinov3' in model.model_identifier
    with torch.no_grad():
        embs = model.get_intermediate_layers(img_tensor, n=N, Dv3=is_dv3)
    embs_np = [utils.to_numpy(emb.squeeze(0)) for emb in embs]
    if channel_last:
        embs_np = [np.transpose(emb_np, (1, 2, 0)) for emb_np in embs_np]

    return embs_np

In [4]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features: dict[ModelTypes, list[list[np.ndarray]]] = {key: [] for key in selected_models}

for key, model in models.items():
    for img_file in image_files:
        img_path = f'{ds_folder}/{img_file}'
        img = Image.open(img_path).convert('RGB')
        feat_list = get_feature_list(model, img, device=DEVICE, channel_last=True, N=n_layers)
        features[key].append(feat_list)

In [5]:
AverageResult: TypeAlias = tuple[np.ndarray, np.ndarray, float, float, np.ndarray]
def average_results(results: list[LinearProbeResult]) -> AverageResult:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [6]:
ramp_types: tuple[RampTypes, ...] = ('lr+ud',)
model_to_layer_results: dict[ModelTypes, list[np.ndarray]] = {key: [] for key in selected_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for key, model in models.items():
    for layer in range(n_layers):
        ramp_scores = np.zeros((len(ramp_types), n_dims))
        for r, ramp in enumerate(ramp_types):
            image_results_for_layer = []
            for i in range(n_imgs):
                feats = features[key][i][layer]
                result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
                image_results_for_layer.append(result)
            averaged = average_results(image_results_for_layer)
            ramp_scores[r] = np.array(averaged[0])
        highest_scores = np.max(ramp_scores, axis=0)
        model_to_layer_results[key].append(highest_scores)

In [7]:
model_to_layer_channel_scores: dict[ModelTypes, np.ndarray] = {key: None for key in selected_models}
for key in selected_models:
    layer_channel_scores = np.array([res for res in model_to_layer_results[key]])
    model_to_layer_channel_scores[key] = layer_channel_scores

In [26]:
%%capture
add_custom_font('resources/fonts', 'Grotesk')
# fig, axs = plt.subplots(1, len(selected_models), figsize=(1 * len(selected_models), 5), sharex=True, sharey=True)

n_rows, n_cols = 2, len(selected_models) + 1
FS = 18
W, H = 4.5, 18

colors: dict[ModelTypes, str] = {
    'dv2': '#5762D5',
    'dv3': '#fcba03',
    'alibi_dv2_coco': '#16ce37',
}


w_spacing = [1/3 for _ in selected_models] + [1/4]
h_spacing = [0.10, 0.6, ]
fig = plt.figure(figsize=(sum([W  * w for w in w_spacing ]), sum([H * h for h in h_spacing])))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, height_ratios=h_spacing, hspace=0.2, wspace=0.8)

h, w = 34, 34



# first_ax = fig.add_subplot(gs[1, 0])
fingerprint_axes = []
for i, key in enumerate(selected_models):
    ax = fig.add_subplot(gs[1, i])
    fingerprint_axes.append(ax)
    arr =  np.array(model_to_layer_results[key])
    ax.imshow(arr.T, aspect='auto', vmin=0, vmax=0.5, cmap='viridis', interpolation='nearest')
    name = model_names[key]
    name = name.replace('(COCO)', '')
    weight = 700 if 'alibi' in key else 500
    ax.set_title(name, fontsize=FS, weight=weight)
    ax.tick_params(labelsize=FS)
    print(i, key)

    if i > 0:
        ax.set_yticks([])
    else:
        ax.set_ylabel('Channel', fontsize=FS)

fingerprint_axes[3].set_xlabel('Layer', fontsize=FS)
cbar_ax = fig.add_subplot(gs[1, -1])
cbar = fig.colorbar(ax.images[0], ax=cbar_ax, fraction=1, pad=0.08,)
cbar.set_label(r'Per-channel $R^2$ scores', fontsize=FS, rotation=270, labelpad=20)
cbar_ax.set_axis_off()
cbar.ax.tick_params(labelsize=FS)

plt.tight_layout()
plt.savefig("saved/S1.5_positional_fingerprints.png", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})